# First benchmarking flow

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1099

In this notebook, we initialize the environment for the first processor benchmarking flow, then we run this flow and display the resulting artifact content.

The goal of this story is to run this flow from the github actions. But we still need this notebook to initialize its environment.

## Initialization

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Choose prefect deployment method
from resources.widget_utils import *
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo() # reload the global vars again
from resources.utils import *  

## Initialize the Prefect variables for mock responses

In [ ]:
# Read the json (copied from https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1099)
with open("1099_first_benchmarking_flow.json", encoding="utf-8") as f:
    mock_template = json.load(f)

# Get all existing processor names
from rs_client.ogcapi.dpr_client import DprProcessor
processor_names = [p.value for p in DprProcessor]
print(f"Processor names: {processor_names}")

# Get all processor versions from the json
processor_versions = sorted(set(
    [v for s in mock_template["scenarios"].values() for v in s["processor_versions"].keys()]
))
print(f"Processor versions: {processor_versions}")

# Get all scenarios from the json
scenario_names = sorted(set(
    [s for s in mock_template["scenarios"].keys()]
))
print(f"Scenarios: {scenario_names}")

# Create one prefect variable for each processor with the json contents
from prefect.variables import Variable
for processor_name in processor_names:
    await Variable.set(
        name=f"benchmarking-{processor_name}-settings",
        value=mock_template, 
        tags=["processing", "all", "benchmarking"], 
        overwrite=True
    )

## Deploy the Prefect flow

In [ ]:
deployed = await deploy_prefect(
    deploy_file="./1099_first_benchmarking_flow.yaml", 
    s3_code_folder=f"users/{OWNER_ID}/code", 
    work_pool_name=os.environ["PREFECT_WORK_POOL_SANDBOX"]
)

## Setup flow parameters

In [ ]:
processor_name = widgets.RadioButtons(
    options=processor_names,
    description="Processor:",
)
display(processor_name)

In [ ]:
processor_version = widgets.RadioButtons(
    options=processor_versions,
    description="Processor version:",
)
display(processor_version)

In [ ]:
scenario_name = widgets.RadioButtons(
    options=scenario_names,
    description="Scenario:",
)
display(scenario_name)

In [ ]:
flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
    "called_by": "notebooks/sprints/sprint38/1099_first_benchmarking_flow/1099_first_benchmarking_flow.ipynb" 
  },
  "processor_name": processor_name.value,
  "processor_version": processor_version.value,
  "scenario_name": scenario_name.value,
}
print(json.dumps(flow_parameters, indent=2))

## Run the Prefect flow

In [ ]:
from rs_workflows.benchmarking.benchmarking_flow import benchmark_processor
state = await run_prefect(
    deploy_name=deployed, 
    py_func=benchmark_processor, 
    params=flow_parameters
)

In [ ]:
# Read the artifact attached to the flow run ID
artifact = prefect_utils.read_artifact(
    http_session, state.state_details.flow_run_id, "benchmarking-result"
)

# Render the artifact as markdown
from IPython.display import Markdown
display(Markdown(artifact))